# 3. Geolocation Enrichment (IP → Country)

In [ ]:
import pandas as pd
fraud_df = pd.read_csv('Fraud_Data.csv')
ip_country_df = pd.read_csv('IpAddress_to_Country.csv')

In [ ]:
def ip_range_to_int(ip_str):
    parts = ip_str.split('.')
    return (int(parts[0]) << 24) + (int(parts[1]) << 16) + (int(parts[2]) << 8) + int(parts[3])

ip_country_df['lower_int'] = ip_country_df['lower_bound_ip_address'].apply(ip_range_to_int)
ip_country_df['upper_int'] = ip_country_df['upper_bound_ip_address'].apply(ip_range_to_int)
ip_country_df.sort_values('lower_int', inplace=True)

In [ ]:
# Convert IP to integer if not already done
def ip_to_int(ip):
    parts = ip.split('.')
    return (int(parts[0]) << 24) + (int(parts[1]) << 16) + (int(parts[2]) << 8) + int(parts[3])

fraud_df['ip_int'] = fraud_df['ip_address'].apply(ip_to_int)
fraud_df_sorted = fraud_df.sort_values('ip_int')

In [ ]:
merged = pd.merge_asof(fraud_df_sorted, ip_country_df[['lower_int','upper_int','country']],
                       left_on='ip_int', right_on='lower_int', direction='forward')
merged = merged[merged['ip_int'] <= merged['upper_int']]
merged.drop(['ip_int','lower_int','upper_int'], axis=1, inplace=True)
fraud_df = merged.sort_index()

In [ ]:
fraud_df['country'].value_counts().head(10)